In [1]:
!pip install transformers peft datasets accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00


In [2]:
import json

# Resume rewriting training examples
training_data = [
    {
        "input": "Worked at a company for 2 years doing software stuff",
        "output": "Software Engineer with 2 years of experience developing "
                  "scalable web applications, improving system performance by 35%",
    },
    {
        "input": "Did customer support and helped people with problems",
        "output": "Customer Support Specialist who resolved 50+ daily tickets, "
                  "maintaining 95% customer satisfaction rate",
    },
    {
        "input": "Made websites using react and other tools",
        "output": "Frontend Developer skilled in React.js, built 10+ responsive "
                  "web applications serving 10,000+ users",
    },
    {
        "input": "Responsible for managing the team and projects",
        "output": "Project Manager led cross-functional team of 8 engineers, "
                  "delivering 5 projects on time within budget",
    },
    {
        "input": "Worked on machine learning and data science projects",
        "output": "ML Engineer developed predictive models achieving 92% accuracy, "
                  "processing 1M+ data points daily using Python and TensorFlow",
    },
    {
        "input": "Did network setup and IT support for company",
        "output": "Network Engineer configured Cisco routers and MikroTik switches "
                  "for 500+ user enterprise network, maintaining 99.9% uptime",
    },
    {
        "input": "Helped with sales and talked to customers",
        "output": "Sales Representative generated $500K annual revenue, "
                  "acquiring 50+ new clients through strategic outreach",
    },
    {
        "input": "Taught students mathematics at school",
        "output": "Mathematics Teacher delivered engaging curriculum to 120+ students, "
                  "improving average test scores by 28% over one semester",
    },
]

# Alpaca format এ convert করো
def format_prompt(example):
    return (
        f"### Instruction:\n"
        f"Rewrite this weak resume bullet point into a strong, "
        f"quantified, ATS-optimized version.\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n{example['output']}"
    )

formatted = [{"text": format_prompt(ex)} for ex in training_data]

with open("resume_train.json", "w") as f:
    json.dump(formatted, f, indent=2)

print(f"✅ Training data: {len(formatted)} examples")
print("\nSample:")
print(formatted[0]["text"])

✅ Training data: 8 examples

Sample:
### Instruction:
Rewrite this weak resume bullet point into a strong, quantified, ATS-optimized version.

### Input:
Worked at a company for 2 years doing software stuff

### Response:
Software Engineer with 2 years of experience developing scalable web applications, improving system performance by 35%


In [3]:
from huggingface_hub import login

from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
print("✅ Model loaded!")

Loading tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading model...


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded!


In [5]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [6]:
from datasets import Dataset

with open("resume_train.json") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )

tokenized = dataset.map(tokenize, batched=True)
print("✅ Dataset ready:", tokenized)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

✅ Dataset ready: Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 8
})


In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./resumevibe-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Training started...")
trainer.train()
print("✅ Training complete!")

Training started...


Step,Training Loss


✅ Training complete!


In [8]:
# Save
model.save_pretrained("./resumevibe-lora")
tokenizer.save_pretrained("./resumevibe-lora")
print("✅ Model saved!")

# Test
def rewrite_bullet(text):
    prompt = (
        f"### Instruction:\n"
        f"Rewrite this weak resume bullet point into a strong, "
        f"quantified, ATS-optimized version.\n\n"
        f"### Input:\n{text}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split("### Response:\n")[-1].strip()


# Test করো
test = "Did Python programming and made some scripts"
print("Input:", test)
print("Output:", rewrite_bullet(test))

✅ Model saved!
Input: Did Python programming and made some scripts
Output: Sure! Here's how you can rewrite this bullet point:

- Did Python programming and created some Python scripts to automate repetitive tasks
- With high programming knowledge and ability to analyze and troubleshoot complex systems

This quantified, ATS-optimized version of the bullet point will make it clear to the hiring manager that you have experience in Python, you have created scripts, and you have the ability to handle complex systems. This will help you stand out


In [9]:
from huggingface_hub import notebook_login

notebook_login()  # HF token দিয়ে login করো

model.push_to_hub("mahabub-unlocked/resumevibe-lora")
tokenizer.push_to_hub("mahabub-unlocked/resumevibe-lora")
print("✅ Pushed to HuggingFace Hub!")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  12%|#2        |  551kB / 4.52MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Pushed to HuggingFace Hub!
